# Notebook 01: Static malware detection — PE headers

**Class 04_01.** Windows PE (Portable Executable) files carry a header with dozens of metadata fields: sizes,
linker and OS versions, section layout, entry point. A scanner can classify a file from these fields alone, without
running it or disassembling it. We build the detector, measure how the benign/malware imbalance hurts it, and
leave the field set wide open for the evasion class (04_03).

In [ ]:
import sys

import numpy as np
import polars as pl

sys.path[:0] = [
    ".",
    "..",
    "../../src",
    "../../../src",
]  # this folder, the course folder (malwlib) and the generic libraries
import aasplot as ap
import malwlib as mw

ap.setup("04")
X, y, cols = mw.load_pe()
print(f"PE header data: {X.shape[0]:,} files x {X.shape[1]} features, {y.mean():.1%} malware")

## The data

Each row is one file; the `Name` column is a hash (`VirusShare_*` for malware). 77 of the 79 columns are numeric
header fields; three quarters of the files are malware, which is nothing like the <1% of a real scanner queue —
we exploit that gap in the imbalance experiment below.

In [ ]:
raw = pl.read_csv(mw.DATA / "benign_malicious_pe_files_train.csv.gz", infer_schema_length=0)
print(raw.select(pl.col("Name").str.starts_with("VirusShare").value_counts(), pl.col("Malware").value_counts()))

## Baseline detector

A random forest on the 77 fields. At this ratio the task is nearly solved — remember this number when the
attack arrives.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
rf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr, ytr)
p = rf.predict_proba(Xte)[:, 1]
mw.eval_bin(yte, p, "RF PE baseline: ")

## Prevalence: the same detector on a realistic queue

The benchmark is 74% malware; a real queue is the opposite. We keep the detector and its threshold fixed and
change only the **malware share** $b$ of the files it scores. Each queue is a **stratified resample** of the
held-out test files: exactly $\mathrm{round}(bN)$ malware and $N-\mathrm{round}(bN)$ benign files, drawn with
replacement inside each class, so class-conditional rates (TPR, FPR) are unchanged and only $b$ moves.
Resampling inside the classes is the correct way to sample here; thinning only the benign training class would push
the data further from a real queue.

In [ ]:
pm, pb = p[yte == 1], p[yte == 0]  # test scores per class
thr0 = 0.5
tpr, fpr = float((pm > thr0).mean()), float((pb > thr0).mean())
print(f"test files: {len(pm)} malware, {len(pb)} benign; at threshold {thr0}: TPR {tpr:.4f}, FPR {fpr:.4f}")

N = 100_000


def queue(b, rng):
    """Stratified resample: round(b N) malware scores and the rest benign scores, with replacement per class."""
    m = round(b * N)
    return rng.choice(pm, m), rng.choice(pb, N - m)


from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
shares = [0.744, 0.3, 0.1, 0.03, 0.01, 0.003, 0.001]
rows = []
for b in shares:
    r = []
    for _ in range(20):
        qm, qb = queue(b, rng)
        tp, fp = int((qm > thr0).sum()), int((qb > thr0).sum())
        prec_q = tp / max(1, tp + fp)
        rec_q = tp / len(qm)
        auc_q = roc_auc_score(np.r_[np.ones(len(qm)), np.zeros(len(qb))], np.r_[qm, qb])
        r.append((auc_q, prec_q, rec_q, 2 * prec_q * rec_q / max(1e-9, prec_q + rec_q), fp))
    auc_m, prec_m, rec_m, f1_m, fp_m = np.mean(r, axis=0)
    formula = tpr * b / (tpr * b + fpr * (1 - b))
    rows.append({"b": b, "auc": auc_m, "precision": prec_m, "recall": rec_m, "f1": f1_m})
    print(
        f"b={b:5.3f}  AUC {auc_m:.4f}  precision {prec_m:.4f} (formula {formula:.4f})  recall {rec_m:.4f}  "
        f"F1 {f1_m:.4f}  false alarms per {N:,} files {fp_m:.0f}"
    )

In [ ]:
f, ax = ap.fig("col", 1.9)
ax.plot(shares, [r["auc"] for r in rows], "-o", color=ap.BLUE, label="AUC")
ax.plot(shares, [r["precision"] for r in rows], "-o", color=ap.RED, label="precision")
ax.plot(shares, [r["recall"] for r in rows], "-o", color=ap.GREEN, label="recall")
ax.set_xscale("log")
ax.set_xlabel("malware share b")
ax.invert_xaxis()
ax.legend()
ap.save(f, "d04_imbalance")

## Worked number and the threshold on the base rate

With the measured TPR and FPR above, the base-rate formula gives the precision of an alarm for any share $b$. At
$b=0.001$ (one malware file per thousand) most alarms are false even though the AUC is the same.

In [ ]:
b0 = 0.001
prec0 = tpr * b0 / (tpr * b0 + fpr * (1 - b0))
print(
    f"worked: TPR {tpr:.4f}, FPR {fpr:.4f}, b {b0}: precision {prec0:.4f}, i.e. 1 true alarm per {1 / prec0:.1f} alarms"
)
rng = np.random.default_rng(1)
qm, qb = queue(0.01, rng)  # b = 1%
thr = np.linspace(0.05, 0.95, 19)
tp = np.array([(qm > t).sum() for t in thr])
fp = np.array([(qb > t).sum() for t in thr])
prec = tp / np.maximum(1, tp + fp)  # P(malware | flagged)
rec = tp / len(qm)
f1 = 2 * prec * rec / np.maximum(1e-9, prec + rec)
print(
    f"b=0.01: best threshold {thr[f1.argmax()]:.2f} -> F1 {f1.max():.3f} (precision {prec[f1.argmax()]:.3f}, recall {rec[f1.argmax()]:.3f}); "
    f"at 0.50: precision {prec[thr.searchsorted(0.5)]:.3f}"
)

In [ ]:
f, ax = ap.fig("col", 1.9)
ax.plot(thr, prec, "-o", color=ap.BLUE, label="precision")
ax.plot(thr, rec, "-o", color=ap.GREEN, label="recall")
ax.plot(thr, f1, "-o", color=ap.RED, label="F1")
ax.set_xlabel("threshold")
ax.legend()
ap.save(f, "d04_threshold")

## What the model looks at

The most important fields are version and size indicators — exactly the fields a packer can rewrite freely
(class 04_03 rewrites them and the detector folds).

In [ ]:
imp = rf.feature_importances_
top = np.argsort(imp)[-10:][::-1]
for i in top[:8]:
    print(f"{cols[i]:34s} {imp[i]:.3f}")

In [ ]:
f, ax = ap.fig("col", 2.2)
ax.barh(np.arange(8), imp[top[:8]][::-1], color=ap.BLUE)
ax.set_yticks(np.arange(8), [c[:24] for c in np.array(cols)[top[:8]]][::-1])
ax.set_xlabel("importance")
ap.save(f, "d04_importance")

## Takeaways

- A 77-field PE header model reaches AUC ≈ .999 — static header detection is *solved* on this benchmark, which
  also makes it a fragile teaching example: the fields are freely malleable.
- The same detector keeps its AUC while precision collapses as the malware share falls; the F1-optimal threshold
  moves with the base rate, so report an operating point on a realistic queue, not AUC alone.
- Top features (`MajorLinkerVersion`, OS/subsystem versions, sizes) are exactly what class 04_03 rewrites.